In [26]:
#1 !pip install -q -U polars sentence-transformers lightgbm snowballstemmer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.2/104.2 kB 4.7 MB/s eta 0:00:00


In [38]:
#2
import os, re, gc, random
import numpy as np
import polars as pl
import torch
from tqdm.auto import tqdm
from scipy.sparse import csr_matrix

DATA_DIR = "/kaggle/input/datasets/absolute1000/avito-data"
TRAIN_PATH = f"{DATA_DIR}/train.parquet"
QUERIES_PATH = f"{DATA_DIR}/benchmark_queries.parquet"
ADS_PATH = f"{DATA_DIR}/benchmark_items.parquet"
INPUT_CACHE = DATA_DIR
CACHE_DIR = "/kaggle/working/avito_cache"
OUT_DIR = "/kaggle/working"
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

N_VAL = 2500
N_RANK = 10000
N_CAND = 250
X_CAND = 50
K_SIM = 50

Q_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
          "search_infm_params_text", "search_category"]
FLOAT_COLS = ["item_rating", "item_rating_reviews_count", "item_price",
              "item_latitude", "item_longitude"]

print(os.listdir(DATA_DIR))

N_BM25 = 100
STOP = {"в", "на", "и", "с", "по", "для", "из", "к", "от", "до", "за", "у", "о", "а", "не", "без", "под", "при"}
N_BM25D = 100
N_DESC = 100

['multilingual-e5-small_train_query.npy', 'multilingual-e5-small_train_filters.npy', 'benchmark_queries.parquet', 'train.parquet', 'benchmark_items.parquet', 'multilingual-e5-small_params.npy']


In [6]:
#3
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-small"
TAG = MODEL_NAME.split("/")[-1]
model = SentenceTransformer(MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
print(model.device)

QCACHE = {}

def encode_raw(texts, prefix):
    return model.encode([prefix + t for t in texts], batch_size=256, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=len(texts) > 1000).astype(np.float32)

def encode(texts, prefix):
    keys = [prefix + t for t in texts]
    missing = sorted({k for k in keys if k not in QCACHE})
    if missing:
        QCACHE.update(zip(missing, encode_raw([k[len(prefix):] for k in missing], prefix)))
    return np.stack([QCACHE[k] for k in keys])

def cached(name, make):
    for d in (CACHE_DIR, INPUT_CACHE):
        path = f"{d}/{name}.npy"
        if os.path.exists(path):
            return np.load(path, allow_pickle=True)
    arr = make()
    np.save(f"{CACHE_DIR}/{name}.npy", arr)
    return arr

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

cuda:0


In [9]:
#4
NEXT_KEYS = [
    "Тип стоимости", "Тип услуги", "Вид услуги", "Начальная цена", "Время для связи",
    "Время работы", "График работы", "Рабочие дни", "Марка", "Название услуги",
    "Своя услуга", "Услуга", "Опыт работы", "Стоимость", "Продолжительность",
    "Как вы работаете", "Куда выезжаете", "Работаете с", "Работа по договору",
    "Готов закупить", "Чем вы", "Выполняю заказы", "Гарантия", "Бригада",
    "Год окончания", "Учебное учреждение", "Берёте ли", "Онлайн-показ",
    "Проживание на объекте", "Предмет или специальность", "Предоплата",
    "Есть портфолио", "Специализация",
]

RATING_RE = re.compile(r"Рейтинг пользователя\s+\d+(?:[.,]\d+)?\s+звезд\w*\s+и\s+выше(?:\s*\[поиск\])?")
RATING_THR_RE = re.compile(r"Рейтинг[^0-9]*(\d(?:[.,]\d)?)")
ADDR_RE = re.compile(
    r"Место оказания услуг.*?(?=\s(?:" + "|".join(map(re.escape, NEXT_KEYS)) + r")(?:\s|,|$)|$)"
)

def clean_spaces(s):
    return re.sub(r"\s+", " ", s).strip()

def cut_rating(s):
    return clean_spaces(RATING_RE.sub(" ", s)) if isinstance(s, str) else ""

def cut_address(s):
    return clean_spaces(ADDR_RE.sub(" ", s)) if isinstance(s, str) else ""

In [12]:
#5
items = (pl.read_parquet(ADS_PATH, columns=[
            "item_id", "item_location_id", "item_category_id", "item_microcat_id",
            "item_is_phone_hidden", "item_is_message_forbidden"] + FLOAT_COLS)
         .with_columns(pl.col(FLOAT_COLS).cast(pl.Float64)))

ITEM_IDS = items["item_id"].to_numpy()
ITEM_LOC = items["item_location_id"].to_numpy()
ITEM_CAT = items["item_category_id"].to_numpy()
ITEM_RATING = items["item_rating"].fill_null(np.nan).to_numpy()
ITEM_CATS = set(ITEM_CAT.tolist())

def corpus_texts(col, fn=None):
    s = pl.read_parquet(ADS_PATH, columns=[col])[col].fill_null("").to_list()
    return [fn(x) for x in s] if fn else s

E_TITLE = cached(f"{TAG}_title", lambda: encode_raw(corpus_texts("item_title_raw"), "passage: "))
E_PARAMS = cached(f"{TAG}_params",
                  lambda: encode_raw(corpus_texts("item_infm_params_text", cut_address), "query: "))
assert len(E_TITLE) == len(E_PARAMS) == len(ITEM_IDS)
print(len(ITEM_IDS), E_TITLE.shape)

Batches:   0%|          | 0/740 [00:00<?, ?it/s]

189212 (189212, 384)


In [28]:
#5.1
import snowballstemmer
from collections import Counter
from scipy.sparse import coo_matrix

STEMMER = snowballstemmer.stemmer("russian")
TOKEN_RE = re.compile(r"[а-яa-z0-9]+")
STEM_CACHE = {}

def tokens(text):
    return TOKEN_RE.findall((text or "").lower().replace("ё", "е"))

def stems(text):
    out = []
    for t in tokens(text):
        s = STEM_CACHE.get(t)
        if s is None:
            s = STEM_CACHE[t] = STEMMER.stemWord(t)
        out.append(s)
    return out

def norm_phrase(text):
    return " " + " ".join(tokens(text)) + " "

texts = pl.read_parquet(ADS_PATH, columns=["item_title_raw", "item_infm_params_text", "item_description_raw"])
titles = texts["item_title_raw"].fill_null("").to_list()
params = [cut_address(s) for s in texts["item_infm_params_text"].fill_null("").to_list()]
descs = texts["item_description_raw"].fill_null("").to_list()
del texts

T_TOK = [stems(s) for s in tqdm(titles)]
P_TOK = [stems(s) for s in tqdm(params)]
T_STEMS = [frozenset(t) for t in T_TOK]
P_STEMS = [frozenset(t) for t in P_TOK]
D_STEMS = [frozenset(stems(s)) for s in tqdm(descs)]
T_NORM = [norm_phrase(s) for s in titles]
D_NORM = [norm_phrase(s) for s in descs]

BM25_K1, BM25_B = 1.2, 0.75
N_DOC = len(ITEM_IDS)
VOCAB, r, c, v = {}, [], [], []
doc_len = np.zeros(N_DOC, dtype=np.float32)
for d in range(N_DOC):
    toks = [s for s in T_TOK[d] + P_TOK[d] if s not in STOP]
    doc_len[d] = len(toks)
    for term, n in Counter(toks).items():
        r.append(d)
        c.append(VOCAB.setdefault(term, len(VOCAB)))
        v.append(n)
r, c, v = np.array(r), np.array(c), np.array(v, dtype=np.float32)
dfreq = np.bincount(c, minlength=len(VOCAB))
idf = np.log1p((N_DOC - dfreq + 0.5) / (dfreq + 0.5)).astype(np.float32)
denom = v + BM25_K1 * (1 - BM25_B + BM25_B * doc_len[r] / doc_len.mean())
W_BM25 = coo_matrix((idf[c] * v * (BM25_K1 + 1) / denom, (r, c)), shape=(N_DOC, len(VOCAB))).tocsc()
del T_TOK, P_TOK, r, c, v, denom

def bm25_scores(q_stems):
    ids = sorted({VOCAB[t] for t in q_stems if t in VOCAB})
    if not ids:
        return np.zeros(N_DOC, dtype=np.float32)
    return np.asarray(W_BM25[:, ids].sum(axis=1)).ravel()

E_DESC = cached(f"{TAG}_desc", lambda: encode_raw([d[:1000] for d in descs], "passage: "))
del titles, params, descs
gc.collect()
print(len(VOCAB), W_BM25.shape, E_DESC.shape)

  0%|          | 0/189212 [00:00<?, ?it/s]

  0%|          | 0/189212 [00:00<?, ?it/s]

  0%|          | 0/189212 [00:00<?, ?it/s]

Batches:   0%|          | 0/740 [00:00<?, ?it/s]

74602 (189212, 74602) (189212, 384)


In [39]:
#5.2
def build_bm25(token_lists, n):
    vocab, r, c, v = {}, [], [], []
    lens = np.zeros(n, dtype=np.float32)
    for d, toks in enumerate(token_lists):
        toks = [s for s in toks if s not in STOP]
        lens[d] = len(toks)
        for term, k in Counter(toks).items():
            r.append(d)
            c.append(vocab.setdefault(term, len(vocab)))
            v.append(k)
    r, c, v = np.array(r), np.array(c), np.array(v, dtype=np.float32)
    dfreq = np.bincount(c, minlength=len(vocab))
    idf = np.log1p((n - dfreq + 0.5) / (dfreq + 0.5)).astype(np.float32)
    denom = v + BM25_K1 * (1 - BM25_B + BM25_B * lens[r] / max(lens.mean(), 1e-6))
    W = coo_matrix((idf[c] * v * (BM25_K1 + 1) / denom, (r, c)), shape=(n, len(vocab))).tocsc()
    return vocab, W

def bm25_with(vocab, W, q_stems):
    ids = sorted({vocab[t] for t in q_stems if t in vocab})
    if not ids:
        return np.zeros(W.shape[0], dtype=np.float32)
    return np.asarray(W[:, ids].sum(axis=1)).ravel()

descs = pl.read_parquet(ADS_PATH, columns=["item_description_raw"])["item_description_raw"].fill_null("").to_list()
VOCAB_D, W_BM25_D = build_bm25((stems(d) for d in tqdm(descs)), len(descs))
del descs
gc.collect()
print(len(VOCAB_D), W_BM25_D.nnz)

  0%|          | 0/189212 [00:00<?, ?it/s]

349504 19375435


In [13]:
#6
def hard_filter(search_category, filters_text):
    if search_category in ITEM_CATS:
        mask = ITEM_CAT == search_category
    else:
        mask = np.ones(len(ITEM_IDS), dtype=bool)
    m = RATING_THR_RE.search(filters_text or "")
    if m:
        thr = float(m.group(1).replace(",", "."))
        mask &= np.isnan(ITEM_RATING) | (ITEM_RATING >= thr)
    return np.flatnonzero(mask)

def split_by_location(search_location_id, allowed):
    local = ITEM_LOC[allowed] == search_location_id
    return allowed[local], allowed[~local]

In [14]:
#7
tr = (pl.read_parquet(TRAIN_PATH, columns=Q_COLS + ["item_id", "item_location_id", "item_microcat_id"])
      .with_columns(pl.col("search_query", "search_infm_params_text").fill_null(""))
      .with_row_index("row"))
groups = tr.select(Q_COLS).unique(maintain_order=True).with_row_index("qid")
tr = tr.join(groups, on=Q_COLS, how="left").sort("row")
print("строк без qid:", tr["qid"].null_count())

TR_SLOC = tr["search_location_id"].to_numpy()
TR_ILOC = tr["item_location_id"].to_numpy()
TR_QID = tr["qid"].to_numpy()

def encode_unique(name, texts, prefix):
    uniq, inv = np.unique(np.array(texts, dtype=object), return_inverse=True)
    return uniq, cached(name, lambda: encode_raw(list(uniq), prefix)), inv

TR_Q_UNIQ, E_TR_Q, TR_Q_INV = encode_unique(f"{TAG}_train_query", tr["search_query"].to_list(), "query: ")
_, E_TR_F, TR_F_INV = encode_unique(f"{TAG}_train_filters",
                                    [cut_rating(s) for s in tr["search_infm_params_text"].to_list()], "query: ")
assert len(TR_Q_UNIQ) == len(E_TR_Q)
print(E_TR_Q.shape, E_TR_F.shape)

строк без qid: 0
(74529, 384) (3519, 384)


In [15]:
#8
elig = (tr.with_columns(pl.col("item_id").is_in(ITEM_IDS.tolist()).alias("in_corpus"))
        .group_by("qid").agg(pl.col("in_corpus").all())
        .filter(pl.col("in_corpus"))["qid"].sort().to_numpy())
print("запросов с ответами в корпусе:", len(elig), "из", groups.height)

rng = np.random.default_rng(SEED)
perm = rng.permutation(elig)
val_qids, rank_qids = perm[:N_VAL], perm[N_VAL:N_VAL + N_RANK]

def make_part(qids):
    qdf = groups.filter(pl.col("qid").is_in(qids.tolist()))
    rel = {q: set(i) for q, i in tr.filter(pl.col("qid").is_in(qids.tolist()))
           .group_by("qid").agg(pl.col("item_id").unique()).iter_rows()}
    return qdf, rel

val_q, VAL_REL = make_part(val_qids)
rank_q, RANK_REL = make_part(rank_qids)

TR_ACTIVE = ~np.isin(TR_QID, np.concatenate([val_qids, rank_qids]))
print("валидация:", val_q.height, "| обучение:", rank_q.height, "| история:", TR_ACTIVE.sum())

запросов с ответами в корпусе: 18436 из 354463
валидация: 2500 | обучение: 10000 | история: 483799


In [41]:
#9
def locations_from_history(search_location_id, q_title, q_filters, x_candidates):
    rows = np.flatnonzero((TR_SLOC == search_location_id) & TR_ACTIVE)
    if len(rows) == 0:
        return []
    prox_query = E_TR_Q[TR_Q_INV[rows]] @ q_title
    prox_filters = (E_TR_F[TR_F_INV[rows]] @ q_filters if q_filters is not None
                    else np.zeros(len(rows), dtype=np.float32))
    order = np.lexsort((-prox_filters, -prox_query))[:x_candidates]
    return np.unique(TR_ILOC[rows[order]]).tolist()

def top_k(scores, k):
    k = min(k, len(scores))
    if k == 0:
        return np.array([], dtype=np.int64)
    return np.argpartition(-scores, k - 1)[:k] if k < len(scores) else np.arange(len(scores))

def top_pos(scores, k):
    t = top_k(scores, k)
    return t[scores[t] > 0]

def search(query_text, filters_text, _LOCAL, _NON_LOCAL, n_candidates,
           search_location_id=None, x_candidates=50):
    q_title = encode([query_text or ""], "query: ")[0]
    filters_text = cut_rating(filters_text)
    q_filters = encode([filters_text], "query: ")[0] if filters_text else None

    if len(_LOCAL) == 0 and search_location_id is not None:
        locs = locations_from_history(search_location_id, q_title, q_filters, x_candidates)
        allowed = np.concatenate([_LOCAL, _NON_LOCAL])
        local = np.isin(ITEM_LOC[allowed], locs)
        _LOCAL, _NON_LOCAL = allowed[local], allowed[~local]

    q_st = [s for s in stems(query_text) if s not in STOP]
    bm = bm25_scores(q_st)
    bmd = bm25_with(VOCAB_D, W_BM25_D, q_st)
    out = {}
    for idx, is_local in ((_LOCAL, True), (_NON_LOCAL, False)):
        if len(idx) == 0:
            continue
        pt = E_TITLE[idx] @ q_title
        pf = E_PARAMS[idx] @ q_filters if q_filters is not None else np.zeros(len(idx), dtype=np.float32)
        dc = E_DESC[idx] @ q_title
        b, bd = bm[idx], bmd[idx]
        pick = np.unique(np.concatenate([
            top_k(pt, n_candidates), top_pos(b, N_BM25), top_pos(bd, N_BM25D), top_k(dc, N_DESC),
        ]))
        for i in pick:
            out[int(idx[i])] = (float(pt[i]), float(pf[i]), float(b[i]), float(bd[i]), float(dc[i]), is_local)
    return out

In [42]:
#10
POOL_COLS = ["qid", "item_id", "prox_title", "prox_filters", "bm25", "bm25_desc", "desc_cos",
             "title_cover", "params_cover", "desc_cover", "phrase_title", "phrase_desc",
             "is_local", "label"]

def warm_cache(qdf):
    encode(qdf["search_query"].to_list(), "query: ")
    f = [x for x in (cut_rating(s) for s in qdf["search_infm_params_text"].to_list()) if x]
    if f:
        encode(f, "query: ")

def build_pool(qdf, rel=None):
    warm_cache(qdf)
    cols = {k: [] for k in POOL_COLS}
    for q in tqdm(qdf.iter_rows(named=True), total=qdf.height):
        allowed = hard_filter(q["search_category"], q["search_infm_params_text"])
        _LOCAL, _NON_LOCAL = split_by_location(q["search_location_id"], allowed)
        res = search(q["search_query"], q["search_infm_params_text"], _LOCAL, _NON_LOCAL,
                     N_CAND, q["search_location_id"], X_CAND)
        qs = {s for s in stems(q["search_query"]) if s not in STOP}
        qn = norm_phrase(q["search_query"])
        has_phrase = qn.strip() != ""
        r = rel[q["qid"]] if rel is not None else set()
        for i, (pt, pf, b, bd, dc, il) in res.items():
            item_id = ITEM_IDS[i]
            cols["qid"].append(q["qid"])
            cols["item_id"].append(item_id)
            cols["prox_title"].append(pt)
            cols["prox_filters"].append(pf)
            cols["bm25"].append(b)
            cols["bm25_desc"].append(bd)
            cols["desc_cos"].append(dc)
            cols["title_cover"].append(len(qs & T_STEMS[i]) / len(qs) if qs else 0.0)
            cols["params_cover"].append(len(qs & P_STEMS[i]) / len(qs) if qs else 0.0)
            cols["desc_cover"].append(len(qs & D_STEMS[i]) / len(qs) if qs else 0.0)
            cols["phrase_title"].append(int(has_phrase and qn in T_NORM[i]))
            cols["phrase_desc"].append(int(has_phrase and qn in D_NORM[i]))
            cols["is_local"].append(il)
            cols["label"].append(int(item_id in r))
    return pl.DataFrame(cols)

def cached_pool(name, qdf, rel=None):
    for d in (CACHE_DIR, INPUT_CACHE):
        path = f"{d}/{name}.parquet"
        if os.path.exists(path):
            return pl.read_parquet(path)
    df = build_pool(qdf, rel)
    df.write_parquet(f"{CACHE_DIR}/{name}.parquet")
    return df

pool_val = cached_pool("pool_val_v4", val_q, VAL_REL)
pool_rank = cached_pool("pool_rank_v4", rank_q, RANK_REL)
print(pool_val.shape, pool_rank.shape, "кандидатов на запрос:", pool_val.height / val_q.height)

  0%|          | 0/2500 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

(1823764, 14) (7308481, 14) кандидатов на запрос: 729.5056


In [1]:
#11
def recall_at(df, rel, score_col=None, n_local=None, n_non=None, k=50):
    if score_col is not None:
        df = df.sort(["qid", score_col], descending=[False, True]).group_by("qid", maintain_order=True).head(k)
    elif n_local is not None:
        df = (df.sort(["qid", "is_local", "prox_title", "prox_filters"], descending=[False, True, True, True])
                .with_columns(pl.int_range(pl.len()).over(["qid", "is_local"]).alias("rk"))
                .filter(pl.when(pl.col("is_local")).then(pl.col("rk") < n_local)
                          .otherwise(pl.col("rk") < n_non)))
    hits = dict(df.group_by("qid").agg(pl.col("label").sum()).iter_rows())
    return np.mean([hits.get(q, 0) / len(items) for q, items in rel.items()])

print("валидация, пул:", recall_at(pool_val, VAL_REL))
print("валидация, 25/25:", recall_at(pool_val, VAL_REL, n_local=25, n_non=25))
print("обучение, пул:", recall_at(pool_rank, RANK_REL))

for d in (CACHE_DIR, INPUT_CACHE):
    p = f"{d}/pool_val_v3.parquet"
    if os.path.exists(p):
        print("пул v3 (без потоков описания):", recall_at(pl.read_parquet(p), VAL_REL))
        break

NameError: name 'pool_val' is not defined

In [32]:
#12
centers = (items.group_by("item_location_id")
           .agg(pl.col("item_latitude").median().alias("c_lat"),
                pl.col("item_longitude").median().alias("c_lon"),
                pl.len().alias("loc_supply"))
           .rename({"item_location_id": "search_location_id"}))
loc_mc_supply = (items.group_by("item_location_id", "item_microcat_id").len()
                 .rename({"item_location_id": "search_location_id", "len": "loc_mc_supply"}))

TR_MC = tr["item_microcat_id"]
mc_vals = np.sort(pl.concat([items["item_microcat_id"], TR_MC]).drop_nulls().unique().to_numpy())
mc2i = {m: i for i, m in enumerate(mc_vals)}

def make_qinfo(qdf):
    return (qdf.select("qid", "search_location_id")
            .join(centers, on="search_location_id", how="left")
            .with_columns(pl.col("c_lat").is_not_null().cast(pl.Int8).alias("had_local"),
                          pl.col("loc_supply").fill_null(0).log1p()))

def make_mc_share(qdf):
    sel = TR_ACTIVE & TR_MC.is_not_null().to_numpy()
    C = csr_matrix((np.ones(sel.sum(), dtype=np.float32),
                    (TR_Q_INV[sel], [mc2i[m] for m in TR_MC.filter(pl.Series(sel)).to_list()])),
                   shape=(len(TR_Q_UNIQ), len(mc_vals)))
    E_q = encode(qdf["search_query"].to_list(), "query: ")
    qids = qdf["qid"].to_numpy()
    parts = []
    for s in tqdm(range(0, len(E_q), 512)):
        S = E_q[s:s + 512] @ E_TR_Q.T
        top = np.argpartition(-S, K_SIM, axis=1)[:, :K_SIM]
        agg = np.vstack([np.asarray(C[t].sum(axis=0)).ravel() for t in top])
        tot = agg.sum(axis=1, keepdims=True)
        share = np.divide(agg, tot, out=np.zeros_like(agg), where=tot > 0)
        qi, mi = np.nonzero(share)
        parts.append(pl.DataFrame({"qid": qids[s:s + 512][qi],
                                   "item_microcat_id": mc_vals[mi], "mc_share": share[qi, mi]}))
    return pl.concat(parts)

R_EARTH = 6371.0

def rank_gap(col, name):
    return [
        pl.col(col).rank("ordinal", descending=True).over(["qid", "is_local"]).alias(f"{name}_rank"),
        (pl.col(col).max().over("qid") - pl.col(col)).alias(f"{name}_gap"),
    ]

def pct(col, name):
    return (pl.col(col).rank("average").over("qid") / pl.col(col).count().over("qid")).alias(name)

def build_features(pool, qdf):
    lat1, lon1 = pl.col("c_lat").radians(), pl.col("c_lon").radians()
    lat2, lon2 = pl.col("item_latitude").radians(), pl.col("item_longitude").radians()
    a = ((lat2 - lat1) / 2).sin() ** 2 + lat1.cos() * lat2.cos() * ((lon2 - lon1) / 2).sin() ** 2
    return (pool
            .join(items.drop("item_location_id", "item_category_id"), on="item_id", how="left")
            .join(make_qinfo(qdf), on="qid", how="left")
            .join(loc_mc_supply, on=["search_location_id", "item_microcat_id"], how="left")
            .join(make_mc_share(qdf), on=["qid", "item_microcat_id"], how="left")
            .with_columns(
                pl.col("mc_share").fill_null(0.0),
                pl.col("loc_mc_supply").fill_null(0).log1p(),
                (2 * R_EARTH * a.sqrt().arcsin()).alias("dist_km"),
                pl.col("item_price").log1p().alias("log_price"),
                pl.col("item_rating_reviews_count").log1p().alias("log_reviews"),
                pct("item_price", "price_pct"),
                pct("item_rating_reviews_count", "reviews_pct"),
                pl.col("is_local").cast(pl.Int8),
                pl.col("item_is_phone_hidden").cast(pl.Int8),
                pl.col("item_is_message_forbidden").cast(pl.Int8),
                ((pl.col("prox_title") - pl.col("prox_title").mean().over("qid"))
                 / pl.col("prox_title").std().over("qid")).fill_nan(0).fill_null(0).alias("prox_title_z"),
                *rank_gap("prox_title", "title"),
                *rank_gap("prox_filters", "filters"),
                *rank_gap("bm25", "bm25"),
                *rank_gap("desc_cos", "desc"),
                *rank_gap("bm25_desc", "bm25d"),
            )
            .drop("c_lat", "c_lon", "item_latitude", "item_longitude", "item_price",
                  "item_rating_reviews_count", "search_location_id"))

feat_val = build_features(pool_val, val_q)
feat_rank = build_features(pool_rank, rank_q)
FEATS = [c for c in feat_val.columns if c not in ("qid", "item_id", "label")]
CAT_FEATS = ["item_microcat_id"]
print(len(FEATS), FEATS)
with pl.Config(tbl_rows=-1):
    print(feat_val.null_count().transpose(include_header=True, column_names=["nulls"]))

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

32 ['prox_title', 'prox_filters', 'bm25', 'desc_cos', 'title_cover', 'params_cover', 'desc_cover', 'phrase_title', 'phrase_desc', 'is_local', 'item_microcat_id', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_rating', 'loc_supply', 'had_local', 'loc_mc_supply', 'mc_share', 'dist_km', 'log_price', 'log_reviews', 'price_pct', 'reviews_pct', 'prox_title_z', 'title_rank', 'title_gap', 'filters_rank', 'filters_gap', 'bm25_rank', 'bm25_gap', 'desc_rank', 'desc_gap']
shape: (35, 2)
┌───────────────────────────┬────────┐
│ column                    ┆ nulls  │
│ ---                       ┆ ---    │
│ str                       ┆ u32    │
╞═══════════════════════════╪════════╡
│ qid                       ┆ 0      │
│ item_id                   ┆ 0      │
│ prox_title                ┆ 0      │
│ prox_filters              ┆ 0      │
│ bm25                      ┆ 0      │
│ desc_cos                  ┆ 0      │
│ title_cover               ┆ 0      │
│ params_cover              ┆ 0      │
│

In [ ]:
#12.1
pos = feat_val.filter(pl.col("label") == 1).with_columns(
    (pl.col("title_rank") <= N_CAND).alias("by_title"),
    ((pl.col("bm25_rank") <= N_BM25) & (pl.col("bm25") > 0)).alias("by_bm25"),
    ((pl.col("bm25d_rank") <= N_BM25D) & (pl.col("bm25_desc") > 0)).alias("by_bm25d"),
    (pl.col("desc_rank") <= N_DESC).alias("by_desc"),
)
print("всего найденных ответов:", pos.height)
print(pos.select(pl.col("by_title", "by_bm25", "by_bm25d", "by_desc").mean()))
only = pos.with_columns((pl.col("by_title").cast(pl.Int8) + pl.col("by_bm25").cast(pl.Int8)
                         + pl.col("by_bm25d").cast(pl.Int8) + pl.col("by_desc").cast(pl.Int8)).alias("n"))
for s in ["by_title", "by_bm25", "by_bm25d", "by_desc"]:
    print(s, "единственный источник:", only.filter((pl.col(s)) & (pl.col("n") == 1)).height)

In [33]:
#13
import lightgbm as lgb

def keep_with_positive(df):
    pos = df.group_by("qid").agg(pl.col("label").max()).filter(pl.col("label") == 1)["qid"].to_list()
    return df.filter(pl.col("qid").is_in(pos))

def to_lgb(df):
    df = df.sort("qid")
    X = df.select(FEATS).to_pandas()
    X[CAT_FEATS] = X[CAT_FEATS].astype("category")
    return X, df["label"].to_numpy(), df.group_by("qid", maintain_order=True).len()["len"].to_numpy()

rank_ids = feat_rank["qid"].unique().sort().to_numpy()
es_ids = np.random.default_rng(SEED).choice(rank_ids, int(len(rank_ids) * 0.1), replace=False).tolist()

X_tr, y_tr, g_tr = to_lgb(keep_with_positive(feat_rank.filter(~pl.col("qid").is_in(es_ids))))
X_es, y_es, g_es = to_lgb(keep_with_positive(feat_rank.filter(pl.col("qid").is_in(es_ids))))

params = dict(objective="lambdarank", metric="ndcg", eval_at=[50], lambdarank_truncation_level=60,
              learning_rate=0.05, num_leaves=63, min_data_in_leaf=200, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, seed=SEED, deterministic=True,
              force_row_wise=True, verbose=-1)

d_tr = lgb.Dataset(X_tr, y_tr, group=g_tr, categorical_feature=CAT_FEATS)
d_es = lgb.Dataset(X_es, y_es, group=g_es, categorical_feature=CAT_FEATS, reference=d_tr)
ranker = lgb.train(params, d_tr, num_boost_round=3000, valid_sets=[d_es],
                   callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])
ranker.save_model(f"{CACHE_DIR}/lgb_ranker_v4.txt", num_iteration=ranker.best_iteration)

Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@50: 0.600033
[200]	valid_0's ndcg@50: 0.611434
[300]	valid_0's ndcg@50: 0.609821
Early stopping, best iteration is:
[256]	valid_0's ndcg@50: 0.612042


In [34]:
#14
def predict(feat):
    X = feat.select(FEATS).to_pandas()
    X[CAT_FEATS] = X[CAT_FEATS].astype("category")
    return feat.with_columns(pl.Series("score", ranker.predict(X, num_iteration=ranker.best_iteration)))

scored_val = predict(feat_val)
print("Recall@50 LightGBM:", recall_at(scored_val, VAL_REL, score_col="score"))

imp = pl.DataFrame({"feature": ranker.feature_name(),
                    "gain": ranker.feature_importance("gain")}).sort("gain", descending=True)
with pl.Config(tbl_rows=-1):
    print(imp)

Recall@50 LightGBM: 0.8964799999999999
shape: (32, 2)
┌───────────────────────────┬───────────────┐
│ feature                   ┆ gain          │
│ ---                       ┆ ---           │
│ str                       ┆ f64           │
╞═══════════════════════════╪═══════════════╡
│ desc_rank                 ┆ 477335.241973 │
│ bm25_rank                 ┆ 134425.859559 │
│ dist_km                   ┆ 124807.055435 │
│ item_microcat_id          ┆ 121888.719347 │
│ mc_share                  ┆ 86772.216541  │
│ is_local                  ┆ 84393.197645  │
│ desc_cover                ┆ 60968.623982  │
│ title_rank                ┆ 55263.301754  │
│ bm25_gap                  ┆ 12737.581384  │
│ reviews_pct               ┆ 12332.184685  │
│ title_cover               ┆ 10823.955198  │
│ desc_gap                  ┆ 9016.714637   │
│ bm25                      ┆ 8562.881013   │
│ item_rating               ┆ 7942.088704   │
│ title_gap                 ┆ 7939.683217   │
│ filters_rank            

In [35]:
#15
TR_ACTIVE[:] = True

bench = (pl.read_parquet(QUERIES_PATH)
         .with_columns(pl.col("search_query", "search_infm_params_text").fill_null(""))
         .with_row_index("qid"))

pool_bench = cached_pool("pool_bench_v4", bench)
feat_bench = build_features(pool_bench, bench)
scored_bench = predict(feat_bench)

top50 = (scored_bench.sort(["qid", "score"], descending=[False, True])
         .group_by("qid", maintain_order=True).head(50)
         .group_by("qid", maintain_order=True).agg(pl.col("item_id")))

answer = (bench.select("qid", "query_id")
          .join(top50, on="qid", how="left")
          .with_columns(pl.col("item_id").fill_null([]).list.join(" ").alias("answer"))
          .select("query_id", "answer"))
answer.write_csv(f"{OUT_DIR}/answer.csv")
answer.write_csv(f"{CACHE_DIR}/answer_v4.csv")

  0%|          | 0/2452 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

In [37]:
#16
chk = pl.read_csv(f"{OUT_DIR}/answer.csv", schema_overrides={"query_id": pl.Utf8, "answer": pl.Utf8})
corpus_ids = set(ITEM_IDS.tolist())

assert chk.columns == ["query_id", "answer"]
assert chk.height == bench.height and chk["query_id"].n_unique() == chk.height
assert set(chk["query_id"]) == set(bench["query_id"])
lens = []
for a in chk["answer"].fill_null(""):
    ids = a.split()
    lens.append(len(ids))
    assert len(ids) <= 50 and len(ids) == len(set(ids))
    assert all(re.fullmatch(r"[0-9a-f]{16}", x) and x in corpus_ids for x in ids)
print("OK | кандидатов на запрос: min", min(lens), "mean", np.mean(lens))

OK | кандидатов на запрос: min 50 mean 50.0
